# 🛠️ Notebook: Introduction to Function Calling

In this notebook we learn how to make LLMs call functions (also called **tool calling**).

## 📚 Sources

- [Ollama: Tool Calling](https://docs.ollama.com/capabilities/tool-calling)
- [OpenAI: Function Calling API](https://platform.openai.com/docs/guides/function-calling)

---

Good luck trying out the Function Calling features! 🤗

## What is Tool Calling?

An LLM can only generate text — it can't look up today's weather, do exact math, or query a database on its own. **Tool calling** closes that gap: you describe a set of functions ("tools") the model is allowed to use, and instead of answering directly, the model can respond with a request to call one of them, with specific arguments. Your code then actually runs that function, and feeds the result back to the model so it can give a final, informed answer.

The model itself never executes any code — it only ever *asks* your program to call a function on its behalf. Running the function, and deciding whether to trust its output, is entirely up to you.

Unlike the previous notebooks, we'll use the **`ollama`** Python package directly in this notebook (instead of the `openai` package) and follow [Ollama's own tool-calling guide](https://docs.ollama.com/capabilities/tool-calling) closely — it has a particularly convenient way of defining tools that we'll use throughout.

One more thing: not every LLM was trained to call tools. `gemma3:4b`, which we used in earlier notebooks, was not. That's why in this notebook we go back to using `LLM_REASONING` (`gemma4:26b`) — its capabilities include tool calling.

In [29]:
LLM_URL = "http://132.199.138.16:11434"
LLM_REASONING = "gemma4:26b"  # the reasoning MoE model - also supports tool calling

In [30]:
from ollama import Client

client = Client(
    host=LLM_URL
)

## 1. Calling a Single Tool

In the structured outputs notebook, we described the shape of data with a **Pydantic class**, and Pydantic converted it into a JSON schema for us. Tools work the same way conceptually, but even more conveniently: the `ollama` package can turn a **plain Python function** directly into a tool definition — no JSON schema to write by hand at all. It reads the function's type hints for the parameter types, and its docstring for the description of the function and each parameter, in the standard "Google style" format shown below (`Args:` section, one line per parameter).

In [31]:
def get_temperature(city: str) -> str:
    """Get the current temperature for a city

    Args:
        city: The name of the city

    Returns:
        The current temperature for the city
    """
    temperatures = {
        "New York": "22°C",
        "London": "15°C",
        "Tokyo": "18°C",
    }
    return temperatures.get(city, "Unknown")

Now let's pass this function directly in the `tools` list of a chat request. `think=True` enables `LLM_REASONING`'s thinking (this is the native Ollama API's own name for what the OpenAI-compatible API called `reasoning_effort` in the previous notebooks — same idea, different parameter name).

In [32]:
messages = [{"role": "user", "content": "What is the temperature in New York?"}]

response = client.chat(model=LLM_REASONING, messages=messages, tools=[get_temperature], think=True)

print("Thinking:", response.message.thinking)
print("Tool calls:", response.message.tool_calls)

Thinking: The user is asking for the temperature in New York. I should check if there's a tool available to get the temperature. The `get_temperature` function seems appropriate for this task. It takes one argument, `city`.

1.  **Identify the intent**: Get temperature.
2.  **Extract parameters**: `city` = "New York".
3.  **Formulate tool call**: `get_temperature(city="New York")`.
Tool calls: [ToolCall(function=Function(name='get_temperature', arguments={'city': 'New York'}))]


The model didn't answer directly — instead, `response.message.tool_calls` tells us it wants to call `get_temperature` with `city="New York"`. Each entry in `tool_calls` has a `.function.name` and `.function.arguments` we can use to actually run the corresponding Python function ourselves.

In [33]:
# Add the model's response to the conversation, so it remembers it asked for a tool call
messages.append(response.message)

if response.message.tool_calls:
    # Simplification: only handle the first tool call for now (Section 2 covers multiple calls)
    call = response.message.tool_calls[0]
    result = get_temperature(**call.function.arguments)  # actually run the function ourselves

    # The result goes back into the conversation as a message with role "tool"
    messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})

    # Ask the model again, now that it has the tool result available
    final_response = client.chat(model=LLM_REASONING, messages=messages, tools=[get_temperature], think=True)
    print(final_response.message.content)

The temperature in New York is 22°C.


What if the model doesn't need a tool at all? Let's ask something completely unrelated to temperatures, with the exact same `tools` list available. The model is free to ignore the tools entirely and just answer directly — in that case, `tool_calls` is simply `None`.

In [34]:
response = client.chat(
    model=LLM_REASONING,
    messages=[{"role": "user", "content": "What is 1+1?"}],
    tools=[get_temperature],
    think=True,
)

print("Tool calls:", response.message.tool_calls)
print("Content:", response.message.content)

Tool calls: None
Content: 1 + 1 = 2


## 2. Parallel Tool Calling

A single request can also trigger **multiple** tool calls at once — for example, if a question needs two different tools, or the same tool for two different inputs. Let's define a second tool and ask about two cities at once.

In [35]:
def get_conditions(city: str) -> str:
    """Get the current weather conditions for a city

    Args:
        city: The name of the city

    Returns:
        The current weather conditions for the city
    """
    conditions = {
        "New York": "Partly cloudy",
        "London": "Rainy",
        "Tokyo": "Sunny",
    }
    return conditions.get(city, "Unknown")

In [36]:
messages = [{
    "role": "user",
    "content": "What are the current weather conditions and temperature in New York and London?",
}]

response = client.chat(
    model=LLM_REASONING,
    messages=messages,
    tools=[get_temperature, get_conditions],  # multiple tools available at once
    think=True,
)
messages.append(response.message)

print(f"The model made {len(response.message.tool_calls)} tool call(s):")
for call in response.message.tool_calls:
    print(f"  {call.function.name}({call.function.arguments})")

The model made 4 tool call(s):
  get_temperature({'city': 'New York'})
  get_conditions({'city': 'New York'})
  get_temperature({'city': 'London'})
  get_conditions({'city': 'London'})


Two cities × two tools = up to four tool calls in a single response. We now execute each one — picking the right Python function by matching `call.function.name` — and add every result back into the conversation before asking for the final answer.

In [37]:
for call in response.message.tool_calls:
    if call.function.name == "get_temperature":
        result = get_temperature(**call.function.arguments)
    elif call.function.name == "get_conditions":
        result = get_conditions(**call.function.arguments)
    else:
        result = "Unknown tool"
    messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})

final_response = client.chat(model=LLM_REASONING, messages=messages, tools=[get_temperature, get_conditions], think=True)
print(final_response.message.content)

In New York, the current temperature is 22°C with partly cloudy conditions. In London, it is 15°C and rainy.


## 3. Multi-Turn Tool Calling (Agent Loop)

So far we always called a tool exactly once and then stopped. But nothing stops the model from calling a tool, looking at the result, and then deciding it needs to call *another* tool before it can answer — for example, to solve a problem that requires several calculation steps in sequence. The general pattern for this is an **agent loop**: keep calling the model and executing whatever tools it asks for, in a `while` loop, until it stops asking for tools and gives a final answer instead.

This is the core mechanism behind what's usually called an "agent": a loop that lets an LLM decide, step by step, which actions to take.

In [38]:
def add(a: int, b: int) -> int:
    """Add two numbers

    Args:
        a: The first number
        b: The second number

    Returns:
        The sum of the two numbers
    """
    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers

    Args:
        a: The first number
        b: The second number

    Returns:
        The product of the two numbers
    """
    return a * b


# A dictionary mapping tool names to the actual Python functions, so we can look
# up and call the right one dynamically based on what the model asks for.
available_functions = {
    "add": add,
    "multiply": multiply,
}

gemma4:26b can't reliably do large multiplications like `23775 * 412` in its head — but if we give it `add` and `multiply` as tools, it can break the problem down into exact steps instead of guessing:

In [39]:
messages = [{"role": "user", "content": "What is (11434+12341)*412?"}]

while True:
    response = client.chat(model=LLM_REASONING, messages=messages, tools=[add, multiply], think=True)
    messages.append(response.message)
    print("Content:", response.message.content)

    if not response.message.tool_calls:
        # No more tool calls -> the model has given its final answer, so we're done
        break

    for tc in response.message.tool_calls:
        if tc.function.name in available_functions:
            result = available_functions[tc.function.name](**tc.function.arguments)
            print(f"  Calling {tc.function.name}({tc.function.arguments}) -> {result}")
            messages.append({"role": "tool", "tool_name": tc.function.name, "content": str(result)})
    # loop continues with the updated messages, until no more tool calls come back

Content: 
  Calling add({'a': 11434, 'b': 12341}) -> 23775
Content: 
  Calling multiply({'a': 23775, 'b': 412}) -> 9795300
Content: The result of (11434 + 12341) * 412 is 9,795,300.


## 4. A Small Shortcut: Passing `available_functions.values()`

If you're already keeping a dictionary of tool name → function for dispatching calls (like `available_functions` in Section 3), you don't need to also maintain a separate list for `tools=` — you can just pass `available_functions.values()` directly.

In [40]:
response = client.chat(
    model=LLM_REASONING,
    messages=[{"role": "user", "content": "What is 17 * 23?"}],
    tools=available_functions.values(),  # same dict from Section 3, no separate list needed
    think=True,
)
print(response.message.tool_calls)

[ToolCall(function=Function(name='multiply', arguments={'a': 17, 'b': 23}))]


## Exercise: A Tool That Calls a Real API

So far our tools only looked up values in a Python dictionary — `get_temperature` above returns fake, hardcoded values. But a tool function can do anything a normal Python function can do, including calling an external API. Let's build a *real* `get_weather` tool that fetches live weather data from [wttr.in](https://wttr.in), a free weather API that needs no API key or sign-up.

Proceed as follows:

1. Implement a function `get_weather(city: str) -> str` with a proper docstring, so `ollama` can turn it into a tool automatically (just like `get_temperature` above) — no manual JSON schema needed.
2. Inside the function, call the wttr.in API (see below for how it works) and return a short summary string.
3. Build the same request → check `tool_calls` → execute → append result → final request workflow from Section 1.

This is how the API works:

In [41]:
import requests

city = "Berlin"  # Also possible: any city name, e.g. Paris, Tokyo, ...
url = f"https://wttr.in/{city}?format=j1"
response = requests.get(url)
data = response.json()

# The current conditions are in the first element of "current_condition"
current = data["current_condition"][0]
print(f"Temperature: {current['temp_C']}°C")
print(f"Condition: {current['weatherDesc'][0]['value']}")
print(f"Humidity: {current['humidity']}%")

# Write your code here ...

Temperature: 21°C
Condition: Sunny
Humidity: 49%


<details>
<summary><b>Show solution</b></summary>

```python
# 1. + 2. Define the tool as a regular Python function with a docstring
def get_weather(city: str) -> str:
    """Retrieves the current weather for a city from a live weather API

    Args:
        city: The name of the city (e.g. Berlin, Paris, Tokyo)

    Returns:
        A short summary of the current weather: temperature, condition, and humidity
    """
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url)
    current = response.json()["current_condition"][0]
    return (
        f"Temperature: {current['temp_C']}°C, "
        f"Condition: {current['weatherDesc'][0]['value']}, "
        f"Humidity: {current['humidity']}%"
    )


# 3. Workflow
messages = [{"role": "user", "content": "What's the weather like in Berlin right now?"}]

response = client.chat(model=LLM_REASONING, messages=messages, tools=[get_weather], think=True)
messages.append(response.message)

if response.message.tool_calls:
    call = response.message.tool_calls[0]
    result = get_weather(**call.function.arguments)
    messages.append({"role": "tool", "tool_name": call.function.name, "content": result})

    final_response = client.chat(model=LLM_REASONING, messages=messages, tools=[get_weather], think=True)
    print(final_response.message.content)
```

</details>